In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv
/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [5]:
import numpy as np
import pandas as pd
import warnings, gc
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics         import roc_auc_score
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import HistGradientBoostingClassifier
from sklearn.neural_network  import MLPClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

# ── Load Data ─────────────────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

try:
    orig = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    orig = orig.drop(columns=['customerID'], errors='ignore')
    orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
    orig['Churn']        = orig['Churn'].map({'Yes':1,'No':0})
    orig['id']           = -1
    print(f'✅ Original: {orig.shape}')
except:
    orig = None

if train['Churn'].dtype == object:
    train['Churn'] = train['Churn'].map({'Yes':1,'No':0})
train['Churn'] = train['Churn'].astype(int)

if orig is not None:
    orig_cols = [c for c in train.columns if c in orig.columns]
    train     = pd.concat([train, orig[orig_cols]], axis=0, ignore_index=True)

print(f'Train: {train.shape} | Test: {test.shape} | Churn: {train["Churn"].mean():.3f}')

TARGET   = 'Churn'
ID_COL   = 'id'
N_SPLITS = 5
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
skf3     = StratifiedKFold(n_splits=3,        shuffle=True, random_state=SEED)

# ── Feature Engineering ───────────────────────────────
def feature_engineering(df, ref_df=None):
    df = df.copy()

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'])

    if 'gender' in df.columns:
        df['gender'] = df['gender'].map({'Male':1,'Female':0}).fillna(0)
    for col in ['Partner','Dependents','PhoneService','PaperlessBilling']:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].map({'Yes':1,'No':0}).fillna(0)
    if 'MultipleLines' in df.columns:
        df['MultipleLines'] = df['MultipleLines'].map(
            {'Yes':2,'No':1,'No phone service':0}).fillna(0)
    for col in ['OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']:
        if col in df.columns:
            df[col] = df[col].map(
                {'Yes':2,'No':1,'No internet service':0}).fillna(0)
    if 'InternetService' in df.columns:
        df['InternetService'] = df['InternetService'].map(
            {'Fiber optic':2,'DSL':1,'No':0}).fillna(0)
    if 'Contract' in df.columns:
        df['Contract'] = df['Contract'].map(
            {'Month-to-month':0,'One year':1,'Two year':2}).fillna(0)
    if 'PaymentMethod' in df.columns:
        df['PaymentMethod'] = df['PaymentMethod'].map(
            {'Electronic check':0,'Mailed check':1,
             'Bank transfer (automatic)':2,
             'Credit card (automatic)':3}).fillna(0)
    for col in df.columns:
        if col in [ID_COL, TARGET]: continue
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Frequency encoding
    if ref_df is not None:
        for col in ['Contract','PaymentMethod','InternetService','tenure']:
            freq = ref_df[col].value_counts(normalize=True)
            df[col+'_freq'] = df[col].map(freq).fillna(0)

    # Charge features
    df['charge_per_tenure']     = df['MonthlyCharges'] / (df['tenure'] + 1)
    df['total_charge_est']      = df['MonthlyCharges'] * df['tenure']
    df['charge_ratio']          = df['TotalCharges'] / (df['total_charge_est'] + 1)
    df['charges_diff']          = df['TotalCharges'] - df['total_charge_est']
    df['charges_diff_pct']      = df['charges_diff'] / (df['total_charge_est'] + 1)
    df['monthly_log']           = np.log1p(df['MonthlyCharges'])
    df['total_log']             = np.log1p(df['TotalCharges'])
    df['tenure_log']            = np.log1p(df['tenure'])
    df['tenure_sqrt']           = np.sqrt(df['tenure'])
    df['tenure_squared']        = df['tenure'] ** 2
    df['monthly_squared']       = df['MonthlyCharges'] ** 2
    df['tenure_x_monthly']      = df['tenure'] * df['MonthlyCharges']

    # Bins
    df['tenure_bin']            = pd.cut(df['tenure'],
        bins=[-1,3,6,12,24,36,48,72,999], labels=range(8)).astype(int)
    df['monthly_bin']           = pd.cut(df['MonthlyCharges'],
        bins=[-1,20,35,50,65,80,95,999], labels=range(7)).astype(int)

    # Flags
    df['is_new']                = (df['tenure'] <= 3).astype(int)
    df['is_very_long']          = (df['tenure'] >= 48).astype(int)
    df['is_high_payer']         = (df['MonthlyCharges'] > 65).astype(int)
    df['is_very_high_payer']    = (df['MonthlyCharges'] > 85).astype(int)
    df['is_low_payer']          = (df['MonthlyCharges'] < 35).astype(int)
    df['is_monthly_contract']   = (df['Contract'] == 0).astype(int)
    df['is_longterm_contract']  = (df['Contract'] == 2).astype(int)
    df['is_fiber']              = (df['InternetService'] == 2).astype(int)
    df['is_no_internet']        = (df['InternetService'] == 0).astype(int)
    df['is_echeck']             = (df['PaymentMethod'] == 0).astype(int)
    df['is_autopay']            = (df['PaymentMethod'] >= 2).astype(int)
    df['has_family']            = ((df['Partner']==1) | (df['Dependents']==1)).astype(int)
    df['no_family']             = ((df['Partner']==0) & (df['Dependents']==0)).astype(int)

    # Service counts
    svc = ['PhoneService','MultipleLines','InternetService',
           'OnlineSecurity','OnlineBackup','DeviceProtection',
           'TechSupport','StreamingTV','StreamingMovies']
    df['num_services']          = df[[c for c in svc if c in df.columns]].clip(0,1).sum(axis=1)
    df['addon_count']           = df[['OnlineSecurity','OnlineBackup','DeviceProtection',
                                      'TechSupport','StreamingTV','StreamingMovies']].clip(0,1).sum(axis=1)
    df['security_count']        = df[['OnlineSecurity','OnlineBackup',
                                      'DeviceProtection','TechSupport']].clip(0,1).sum(axis=1)
    df['streaming_count']       = df[['StreamingTV','StreamingMovies']].clip(0,1).sum(axis=1)

    # Risk & loyalty
    df['risk_score']            = (
        df['is_monthly_contract']            * 4 +
        df['is_fiber']                       * 3 +
        df['is_echeck']                      * 2 +
        df['is_new']                         * 2 +
        df['is_high_payer']                  * 1 +
        df['SeniorCitizen']                  * 1 +
        df['no_family']                      * 1 +
        (1 - df['OnlineSecurity'].clip(0,1)) * 1 +
        (1 - df['TechSupport'].clip(0,1))    * 1
    )
    df['loyalty_score']         = (
        df['is_longterm_contract']           * 4 +
        df['is_autopay']                     * 2 +
        df['is_very_long']                   * 3 +
        df['OnlineSecurity'].clip(0,1)       * 1 +
        df['TechSupport'].clip(0,1)          * 1 +
        df['has_family']                     * 1
    )
    df['churn_risk_index']      = df['risk_score'] - df['loyalty_score']
    df['risk_x_monthly']        = df['risk_score']    * df['MonthlyCharges']
    df['loyalty_x_tenure']      = df['loyalty_score'] * df['tenure']

    # Cross features
    df['fiber_x_monthly']       = df['is_fiber']           * df['MonthlyCharges']
    df['fiber_x_echeck']        = df['is_fiber']           * df['is_echeck']
    df['fiber_x_no_contract']   = df['is_fiber']           * df['is_monthly_contract']
    df['fiber_x_no_security']   = df['is_fiber']           * (1-df['OnlineSecurity'].clip(0,1))
    df['new_x_fiber']           = df['is_new']             * df['is_fiber']
    df['new_x_echeck']          = df['is_new']             * df['is_echeck']
    df['new_x_monthly']         = df['is_new']             * df['MonthlyCharges']
    df['senior_x_fiber']        = df['SeniorCitizen']      * df['is_fiber']
    df['senior_x_monthly']      = df['SeniorCitizen']      * df['MonthlyCharges']
    df['contract_x_tenure']     = df['Contract']           * df['tenure']
    df['contract_x_monthly']    = df['Contract']           * df['MonthlyCharges']
    df['paperless_x_echeck']    = df['PaperlessBilling']   * df['is_echeck']
    df['no_support']            = ((df['OnlineSecurity']==1) & (df['TechSupport']==1)).astype(int)
    df['full_security']         = ((df['OnlineSecurity']==2) & (df['TechSupport']==2) &
                                   (df['DeviceProtection']==2) & (df['OnlineBackup']==2)).astype(int)
    df['charge_per_service']    = df['MonthlyCharges'] / (df['num_services'] + 1)
    df['fiber_echeck_new']      = df['is_fiber'] * df['is_echeck'] * df['is_new']
    df['risk_tenure_monthly']   = df['risk_score'] * df['tenure_log'] * df['monthly_log']

    # Agg stats
    num_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c not in [ID_COL, TARGET]]
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    return df

ref_df = train.copy()
ref_df['TotalCharges'] = pd.to_numeric(ref_df['TotalCharges'], errors='coerce').fillna(0)
for col in ['Contract','InternetService','PaymentMethod']:
    if ref_df[col].dtype == object:
        ref_df[col] = LabelEncoder().fit_transform(ref_df[col].astype(str))

train_fe = feature_engineering(train, ref_df)
test_fe  = feature_engineering(test,  ref_df)

FEATURES = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X        = train_fe[FEATURES]
y        = train_fe[TARGET]
X_test   = test_fe[FEATURES].reindex(columns=FEATURES, fill_value=0)

print(f'✅ Features: {len(FEATURES)} | X: {X.shape} | X_test: {X_test.shape}')

✅ Original: (7043, 21)
Train: (601237, 21) | Test: (254655, 20) | Churn: 0.226
✅ Features: 80 | X: (601237, 80) | X_test: (254655, 80)


In [9]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

Sun Mar 22 08:51:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             40W /  250W |     339MiB /  16384MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# ── Tune XGBoost ──────────────────────────────────────
print('🔍 Tuning XGBoost (30 trials)...')

def xgb_obj(trial):
    p = dict(
        n_estimators          = trial.suggest_int('ne', 500, 2000),
        learning_rate         = trial.suggest_float('lr', 0.01, 0.08, log=True),
        max_depth             = trial.suggest_int('md', 4, 7),
        min_child_weight      = trial.suggest_int('mcw', 3, 20),
        gamma                 = trial.suggest_float('gm', 0.0, 1.0),
        subsample             = trial.suggest_float('ss', 0.6, 1.0),
        colsample_bytree      = trial.suggest_float('cs', 0.5, 1.0),
        reg_alpha             = trial.suggest_float('ra', 0.0, 3.0),
        reg_lambda            = trial.suggest_float('rl', 0.5, 4.0),
        scale_pos_weight      = (y==0).sum()/(y==1).sum(),
        tree_method='hist', device='cuda', eval_metric='auc',
        early_stopping_rounds=50, random_state=SEED, n_jobs=-1
    )
    scores = []
    for ti, vi in skf3.split(X, y):
        m = XGBClassifier(**p)
        m.fit(X.iloc[ti], y.iloc[ti],
              eval_set=[(X.iloc[vi], y.iloc[vi])], verbose=False)
        scores.append(roc_auc_score(y.iloc[vi],
                      m.predict_proba(X.iloc[vi])[:,1]))
    return np.mean(scores)

s1 = optuna.create_study(direction='maximize',
     sampler=optuna.samplers.TPESampler(seed=SEED))
s1.optimize(xgb_obj, n_trials=30, show_progress_bar=True)
p = s1.best_params
best_xgb = dict(
    n_estimators=p['ne'], learning_rate=p['lr'],
    max_depth=p['md'], min_child_weight=p['mcw'],
    gamma=p['gm'], subsample=p['ss'],
    colsample_bytree=p['cs'], reg_alpha=p['ra'], reg_lambda=p['rl'],
    scale_pos_weight=(y==0).sum()/(y==1).sum(),
    tree_method='hist', device='cuda', eval_metric='auc',
    early_stopping_rounds=100, random_state=SEED, n_jobs=-1
)
print(f'✅ XGB best AUC: {s1.best_value:.5f}')
print(f'   Params: {s1.best_params}')

# ── Tune LightGBM ─────────────────────────────────────
print('\n🔍 Tuning LightGBM (30 trials)...')

def lgb_obj(trial):
    p = dict(
        n_estimators      = trial.suggest_int('ne', 500, 2000),
        learning_rate     = trial.suggest_float('lr', 0.01, 0.08, log=True),
        num_leaves        = trial.suggest_int('nl', 20, 100),
        max_depth         = trial.suggest_int('md', 3, 8),
        min_child_samples = trial.suggest_int('mcs', 10, 80),
        feature_fraction  = trial.suggest_float('ff', 0.5, 1.0),
        bagging_fraction  = trial.suggest_float('bf', 0.5, 1.0),
        bagging_freq      = trial.suggest_int('bfq', 1, 7),
        reg_alpha         = trial.suggest_float('ra', 0.0, 3.0),
        reg_lambda        = trial.suggest_float('rl', 0.0, 3.0),
        is_unbalance=True, device='gpu', gpu_platform_id=0,
        gpu_device_id=0, random_state=SEED, n_jobs=-1, verbosity=-1
    )
    scores = []
    for ti, vi in skf3.split(X, y):
        m = LGBMClassifier(**p)
        m.fit(X.iloc[ti], y.iloc[ti],
              eval_set=[(X.iloc[vi], y.iloc[vi])], eval_metric='auc',
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(False)])
        scores.append(roc_auc_score(y.iloc[vi],
                      m.predict_proba(X.iloc[vi])[:,1]))
    return np.mean(scores)

s2 = optuna.create_study(direction='maximize',
     sampler=optuna.samplers.TPESampler(seed=SEED))
s2.optimize(lgb_obj, n_trials=30, show_progress_bar=True)
p2 = s2.best_params
best_lgb = dict(
    n_estimators=p2['ne'], learning_rate=p2['lr'],
    num_leaves=p2['nl'], max_depth=p2['md'],
    min_child_samples=p2['mcs'], feature_fraction=p2['ff'],
    bagging_fraction=p2['bf'], bagging_freq=p2['bfq'],
    reg_alpha=p2['ra'], reg_lambda=p2['rl'],
    is_unbalance=True, device='gpu', gpu_platform_id=0,
    gpu_device_id=0, random_state=SEED, n_jobs=-1, verbosity=-1
)
print(f'✅ LGB best AUC: {s2.best_value:.5f}')
print(f'   Params: {s2.best_params}')

# ── Tune CatBoost ─────────────────────────────────────
print('\n🔍 Tuning CatBoost (20 trials)...')

def cat_obj(trial):
    p = dict(
        iterations         = trial.suggest_int('it', 500, 2000),
        learning_rate      = trial.suggest_float('lr', 0.01, 0.08, log=True),
        depth              = trial.suggest_int('d', 4, 7),
        l2_leaf_reg        = trial.suggest_float('l2', 1.0, 8.0),
        bootstrap_type     = 'Bernoulli',
        subsample          = trial.suggest_float('ss', 0.6, 1.0),
        min_data_in_leaf   = trial.suggest_int('mdl', 10, 80),
        auto_class_weights='Balanced', eval_metric='AUC',
        task_type='GPU', random_seed=SEED, verbose=False
    )
    scores = []
    for ti, vi in skf3.split(X, y):
        m = CatBoostClassifier(**p)
        m.fit(X.iloc[ti], y.iloc[ti],
              eval_set=(X.iloc[vi], y.iloc[vi]),
              early_stopping_rounds=50)
        scores.append(roc_auc_score(y.iloc[vi],
                      m.predict_proba(X.iloc[vi])[:,1]))
    return np.mean(scores)

s3 = optuna.create_study(direction='maximize',
     sampler=optuna.samplers.TPESampler(seed=SEED))
s3.optimize(cat_obj, n_trials=20, show_progress_bar=True)
p3 = s3.best_params
best_cat = dict(
    iterations=p3['it'], learning_rate=p3['lr'],
    depth=p3['d'], l2_leaf_reg=p3['l2'],
    bootstrap_type='Bernoulli', subsample=p3['ss'],
    min_data_in_leaf=p3['mdl'],
    auto_class_weights='Balanced', eval_metric='AUC',
    task_type='GPU', random_seed=SEED, verbose=False
)
print(f'✅ CAT best AUC: {s3.best_value:.5f}')
print(f'   Params: {s3.best_params}')

print('\n' + '='*50)
print('✅ TUNING COMPLETE')
print('='*50)
print(f'  XGB best : {s1.best_value:.5f}')
print(f'  LGB best : {s2.best_value:.5f}')
print(f'  CAT best : {s3.best_value:.5f}')

🔍 Tuning XGBoost (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

✅ XGB best AUC: 0.91525
   Params: {'ne': 1804, 'lr': 0.051273598376649326, 'md': 5, 'mcw': 20, 'gm': 0.9937414698072071, 'ss': 0.9402468530565209, 'cs': 0.5755605734667852, 'ra': 2.0035133150901765, 'rl': 3.967932980027521}

🔍 Tuning LightGBM (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

✅ LGB best AUC: 0.91172
   Params: {'ne': 1808, 'lr': 0.030714447227463495, 'nl': 96, 'md': 8, 'mcs': 59, 'ff': 0.5001310561523564, 'bf': 0.5110021684815566, 'bfq': 2, 'ra': 0.9403014471847034, 'rl': 2.3346932591058907}

🔍 Tuning CatBoost (20 trials)...


  0%|          | 0/20 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric perio

✅ CAT best AUC: 0.91491
   Params: {'it': 1697, 'lr': 0.03736759455496556, 'd': 5, 'l2': 3.419856216737866, 'ss': 0.9086656573027754, 'mdl': 58}

✅ TUNING COMPLETE
  XGB best : 0.91525
  LGB best : 0.91172
  CAT best : 0.91491


In [10]:
# ── Init OOF arrays ───────────────────────────────────
oof_xgb = np.zeros(len(X)); pred_xgb = np.zeros(len(X_test))
oof_lgb = np.zeros(len(X)); pred_lgb = np.zeros(len(X_test))
oof_cat = np.zeros(len(X)); pred_cat = np.zeros(len(X_test))
oof_hgb = np.zeros(len(X)); pred_hgb = np.zeros(len(X_test))
oof_mlp = np.zeros(len(X)); pred_mlp = np.zeros(len(X_test))

# ── XGBoost ───────────────────────────────────────────
print('='*50)
print('🚀 XGBoost (5-fold)...')
print('='*50)
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    m = XGBClassifier(**best_xgb)
    m.fit(X.iloc[ti], y.iloc[ti],
          eval_set=[(X.iloc[vi], y.iloc[vi])], verbose=False)
    oof_xgb[vi]  = m.predict_proba(X.iloc[vi])[:,1]
    pred_xgb    += m.predict_proba(X_test)[:,1] / N_SPLITS
    print(f'  Fold {fold+1} | AUC: {roc_auc_score(y.iloc[vi], oof_xgb[vi]):.5f}')
print(f'📊 XGBoost: {roc_auc_score(y, oof_xgb):.5f}')

# ── LightGBM ──────────────────────────────────────────
print('\n' + '='*50)
print('🚀 LightGBM (5-fold)...')
print('='*50)
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    m = LGBMClassifier(**best_lgb)
    m.fit(X.iloc[ti], y.iloc[ti],
          eval_set=[(X.iloc[vi], y.iloc[vi])], eval_metric='auc',
          callbacks=[lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(False)])
    oof_lgb[vi]  = m.predict_proba(X.iloc[vi])[:,1]
    pred_lgb    += m.predict_proba(X_test)[:,1] / N_SPLITS
    print(f'  Fold {fold+1} | AUC: {roc_auc_score(y.iloc[vi], oof_lgb[vi]):.5f}')
print(f'📊 LightGBM: {roc_auc_score(y, oof_lgb):.5f}')

# ── CatBoost ──────────────────────────────────────────
print('\n' + '='*50)
print('🚀 CatBoost (5-fold)...')
print('='*50)
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    m = CatBoostClassifier(**best_cat)
    m.fit(X.iloc[ti], y.iloc[ti],
          eval_set=(X.iloc[vi], y.iloc[vi]),
          early_stopping_rounds=100)
    oof_cat[vi]  = m.predict_proba(X.iloc[vi])[:,1]
    pred_cat    += m.predict_proba(X_test)[:,1] / N_SPLITS
    print(f'  Fold {fold+1} | AUC: {roc_auc_score(y.iloc[vi], oof_cat[vi]):.5f}')
print(f'📊 CatBoost: {roc_auc_score(y, oof_cat):.5f}')

# ── HistGradientBoosting ──────────────────────────────
print('\n' + '='*50)
print('🚀 HistGradientBoosting (5-fold)...')
print('='*50)
for fold, (ti, vi) in enumerate(skf.split(X, y)):
    m = HistGradientBoostingClassifier(
        max_iter=1000, learning_rate=0.05,
        max_depth=6, min_samples_leaf=20,
        l2_regularization=0.1,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=30,
        random_state=SEED
    )
    m.fit(X.iloc[ti], y.iloc[ti])
    oof_hgb[vi]  = m.predict_proba(X.iloc[vi])[:,1]
    pred_hgb    += m.predict_proba(X_test)[:,1] / N_SPLITS
    print(f'  Fold {fold+1} | AUC: {roc_auc_score(y.iloc[vi], oof_hgb[vi]):.5f}')
print(f'📊 HGB: {roc_auc_score(y, oof_hgb):.5f}')

# ── MLP (sklearn) ─────────────────────────────────────
print('\n' + '='*50)
print('🚀 MLP Neural Network (5-fold, CPU)...')
print('='*50)
sc       = StandardScaler()
X_sc     = sc.fit_transform(X)
Xtest_sc = sc.transform(X_test)

for fold, (ti, vi) in enumerate(skf.split(X_sc, y)):
    m = MLPClassifier(
        hidden_layer_sizes  = (256, 128, 64, 32),
        activation          = 'relu',
        solver              = 'adam',
        learning_rate_init  = 5e-4,
        batch_size          = 2048,
        max_iter            = 200,
        early_stopping      = True,
        validation_fraction = 0.1,
        n_iter_no_change    = 15,
        random_state        = SEED
    )
    m.fit(X_sc[ti], y.iloc[ti])
    oof_mlp[vi]  = m.predict_proba(X_sc[vi])[:,1]
    pred_mlp    += m.predict_proba(Xtest_sc)[:,1] / N_SPLITS
    print(f'  Fold {fold+1} | AUC: {roc_auc_score(y.iloc[vi], oof_mlp[vi]):.5f}')
print(f'📊 MLP: {roc_auc_score(y, oof_mlp):.5f}')

# ── Summary ───────────────────────────────────────────
all_names = ['XGBoost','LightGBM','CatBoost','HGB','MLP']
all_oofs  = [oof_xgb, oof_lgb, oof_cat, oof_hgb, oof_mlp]
all_preds = [pred_xgb, pred_lgb, pred_cat, pred_hgb, pred_mlp]

print('\n' + '='*50)
print('📊 ALL MODELS SUMMARY')
print('='*50)
for n, o in zip(all_names, all_oofs):
    print(f'  {n:12s}: {roc_auc_score(y, o):.5f}')

🚀 XGBoost (5-fold)...
  Fold 1 | AUC: 0.91583
  Fold 2 | AUC: 0.91621
  Fold 3 | AUC: 0.91403
  Fold 4 | AUC: 0.91504
  Fold 5 | AUC: 0.91614
📊 XGBoost: 0.91544

🚀 LightGBM (5-fold)...
  Fold 1 | AUC: 0.91250
  Fold 2 | AUC: 0.91231
  Fold 3 | AUC: 0.91058
  Fold 4 | AUC: 0.91147
  Fold 5 | AUC: 0.91221
📊 LightGBM: 0.91178

🚀 CatBoost (5-fold)...


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 1 | AUC: 0.91532


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 2 | AUC: 0.91567


...

  Fold 3 | AUC: 0.91363


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 4 | AUC: 0.91483


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 5 | AUC: 0.91569
📊 CatBoost: 0.91502

🚀 HistGradientBoosting (5-fold)...
  Fold 1 | AUC: 0.91552
  Fold 2 | AUC: 0.91584
  Fold 3 | AUC: 0.91363
  Fold 4 | AUC: 0.91486
  Fold 5 | AUC: 0.91567
📊 HGB: 0.91509

🚀 MLP Neural Network (5-fold, CPU)...
  Fold 1 | AUC: 0.91303
  Fold 2 | AUC: 0.91242
  Fold 3 | AUC: 0.91104
  Fold 4 | AUC: 0.91226
  Fold 5 | AUC: 0.91254
📊 MLP: 0.91210

📊 ALL MODELS SUMMARY
  XGBoost     : 0.91544
  LightGBM    : 0.91178
  CatBoost    : 0.91502
  HGB         : 0.91509
  MLP         : 0.91210


In [11]:
oof_matrix  = np.column_stack(all_oofs)
pred_matrix = np.column_stack(all_preds)

# ── Optuna Blend ──────────────────────────────────────
print('🔍 Optuna blend (2000 trials)...')

def blend_obj(trial):
    w = np.array([trial.suggest_float(f'w{i}', 0.0, 1.0)
                  for i in range(len(all_names))])
    w = w / w.sum()
    return roc_auc_score(y, (oof_matrix * w).sum(axis=1))

bs = optuna.create_study(direction='maximize',
     sampler=optuna.samplers.TPESampler(seed=SEED))
bs.optimize(blend_obj, n_trials=2000, show_progress_bar=True)

best_w     = np.array([bs.best_params[f'w{i}'] for i in range(len(all_names))])
best_w     = best_w / best_w.sum()
oof_blend  = (oof_matrix  * best_w).sum(axis=1)
pred_blend = (pred_matrix * best_w).sum(axis=1)

print('\n⚖️  Weights:')
for n, w in zip(all_names, best_w):
    print(f'  {n:12s}: {w:.4f}')
print(f'\n📊 Blend OOF: {roc_auc_score(y, oof_blend):.5f}')

# ── XGB Meta-Stacker ──────────────────────────────────
print('\n🚀 XGB Meta-Stacker...')
oof_stack  = np.zeros(len(y))
pred_stack = np.zeros(len(X_test))

for ti, vi in skf.split(oof_matrix, y):
    meta = XGBClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=3, subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist', device='cuda',
        eval_metric='auc',
        early_stopping_rounds=30,
        random_state=SEED
    )
    meta.fit(oof_matrix[ti], y.iloc[ti],
             eval_set=[(oof_matrix[vi], y.iloc[vi])],
             verbose=False)
    oof_stack[vi] = meta.predict_proba(oof_matrix[vi])[:,1]
    pred_stack   += meta.predict_proba(pred_matrix)[:,1] / N_SPLITS

print(f'📊 Stack OOF: {roc_auc_score(y, oof_stack):.5f}')

# ── LR Meta ───────────────────────────────────────────
print('\n🚀 LR Meta...')
oof_lr  = np.zeros(len(y))
pred_lr = np.zeros(len(X_test))
mm      = np.column_stack([oof_blend, oof_stack])
pm      = np.column_stack([pred_blend, pred_stack])

for ti, vi in skf.split(mm, y):
    lr = LogisticRegression(C=1.0, max_iter=1000)
    lr.fit(mm[ti], y.iloc[ti])
    oof_lr[vi] = lr.predict_proba(mm[vi])[:,1]
    pred_lr   += lr.predict_proba(pm)[:,1] / N_SPLITS

print(f'📊 LR Meta OOF: {roc_auc_score(y, oof_lr):.5f}')

# ── Pick Best ─────────────────────────────────────────
results = {
    'Blend'  : (roc_auc_score(y, oof_blend),  pred_blend),
    'Stack'  : (roc_auc_score(y, oof_stack),  pred_stack),
    'LR_Meta': (roc_auc_score(y, oof_lr),     pred_lr),
    'B+S_avg': (roc_auc_score(y, 0.5*oof_blend+0.5*oof_stack),
                0.5*pred_blend+0.5*pred_stack),
}
best_key   = max(results, key=lambda k: results[k][0])
pred_final = results[best_key][1]

sub['Churn'] = pred_final
sub.to_csv('submission1.csv', index=False)

print('\n' + '='*50)
print('🏆 FINAL SUMMARY')
print('='*50)
for n, o in zip(all_names, all_oofs):
    print(f'  {n:12s}: {roc_auc_score(y, o):.5f}')
print(f'  {"─"*32}')
for k, (auc, _) in results.items():
    tag = ' ← BEST' if k == best_key else ''
    print(f'  {k:12s}: {auc:.5f}{tag}')
print('='*50)
print(f'✅ Submit! | Mean: {pred_final.mean():.4f} | Std: {pred_final.std():.4f}')

🔍 Optuna blend (2000 trials)...


  0%|          | 0/2000 [00:00<?, ?it/s]


⚖️  Weights:
  XGBoost     : 0.5832
  LightGBM    : 0.0000
  CatBoost    : 0.2098
  HGB         : 0.2067
  MLP         : 0.0003

📊 Blend OOF: 0.91553

🚀 XGB Meta-Stacker...
📊 Stack OOF: 0.91542

🚀 LR Meta...
📊 LR Meta OOF: 0.91552

🏆 FINAL SUMMARY
  XGBoost     : 0.91544
  LightGBM    : 0.91178
  CatBoost    : 0.91502
  HGB         : 0.91509
  MLP         : 0.91210
  ────────────────────────────────
  Blend       : 0.91553 ← BEST
  Stack       : 0.91542
  LR_Meta     : 0.91552
  B+S_avg     : 0.91552
✅ Submit! | Mean: 0.3164 | Std: 0.3291


In [12]:
from sklearn.isotonic import IsotonicRegression

# Calibrate using OOF
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_blend, y)
pred_calibrated = iso.predict(pred_blend)

print(f'Before : mean={pred_blend.mean():.4f}')
print(f'After  : mean={pred_calibrated.mean():.4f}')
print(f'Target : mean={y.mean():.4f}')

# Save both
sub['Churn'] = pred_blend
sub.to_csv('submission_original.csv', index=False)

sub['Churn'] = pred_calibrated
sub.to_csv('submission_calibrated.csv', index=False)

print('✅ Both saved — submit both and compare!')

Before : mean=0.3164
After  : mean=0.2185
Target : mean=0.2257
✅ Both saved — submit both and compare!
